In [70]:
# %% [markdown]
# Phase 2.1 — NeurIPS Open Polymer Prediction 2025
# Cell 1 — Setup & Imports

# %% 
import os, sys, gc, math, warnings, json, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

# Chemistry
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # silence RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

# ML
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer

# LightGBM
try:
    import lightgbm as lgb
except Exception as e:
    raise RuntimeError("LightGBM is required. Try: pip install lightgbm") from e

# Mordred (optional; can be slow)
MORDRED_ENABLED = True
try:
    if MORDRED_ENABLED:
        from mordred import Calculator, descriptors
except Exception:
    MORDRED_ENABLED = False  # auto-disable if not available

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# File paths — adjust if needed
DATA_DIR = "./"  # point to your data directory
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

# Sanity checks (Cell 1)
print(f"✅ Imports OK | Mordred enabled: {MORDRED_ENABLED}")
print(f"Using DATA_DIR='{DATA_DIR}'. Expect train/test at:\n  {TRAIN_PATH}\n  {TEST_PATH}")


✅ Imports OK | Mordred enabled: True
Using DATA_DIR='./'. Expect train/test at:
  ./train.csv
  ./test.csv


In [71]:
# %%
warnings.filterwarnings("ignore")

# Load
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# Normalize column names: common gotchas include 'Id' vs 'id'
train.columns = [c.strip() for c in train.columns]
test.columns  = [c.strip() for c in test.columns]

# Coerce 'id' to lowercase and make sure it exists
if "Id" in train.columns and "id" not in train.columns:
    train = train.rename(columns={"Id":"id"})
if "Id" in test.columns and "id" not in test.columns:
    test = test.rename(columns={"Id":"id"})

# Required columns
REQUIRED = ["id", "SMILES"]
missing_train = [c for c in REQUIRED if c not in train.columns]
missing_test  = [c for c in REQUIRED if c not in test.columns]
assert not missing_train, f"Missing in train: {missing_train}"
assert not missing_test,  f"Missing in test: {missing_test}"

print("✅ Loaded data")
print("train:", train.shape, "test:", test.shape)
print("train columns:", list(train.columns)[:12], "...")


✅ Loaded data
train: (7973, 7) test: (3, 2)
train columns: ['id', 'SMILES', 'Tg', 'FFV', 'Tc', 'Density', 'Rg'] ...


In [72]:
# %%
warnings.filterwarnings("ignore")

EXPECTED_TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]  # canonical order

# Find targets present in train
targets = [t for t in EXPECTED_TARGETS if t in train.columns]
assert set(targets) == set(EXPECTED_TARGETS), \
    f"Targets missing from train: {set(EXPECTED_TARGETS)-set(targets)}"

# Guardrails: ensure `id` not part of targets; no accidental extra numeric fields
assert "id" in train.columns, "Expected 'id' column not found."
assert not any(t.lower()=="id" for t in targets), "‘id’ leaked into targets!"

extra_numeric = [c for c in train.select_dtypes(include=[np.number]).columns
                 if c not in (["id"] + EXPECTED_TARGETS)]
# We won't use any raw extra numeric columns from train to avoid leakage
if extra_numeric:
    print("ℹ️ Note: ignoring extra numeric columns in train:", extra_numeric)

print("✅ Targets locked:", targets)


✅ Targets locked: ['Tg', 'FFV', 'Tc', 'Density', 'Rg']


In [81]:
# %% 
# Patch A — Target NaN Audit
nan_counts = {t: int(train[t].isna().sum()) for t in EXPECTED_TARGETS}
n_rows = len(train)
print("🧪 Label NaN audit:")
for t, k in nan_counts.items():
    print(f"  - {t}: {k} / {n_rows} missing ({k/n_rows:.2%})")

# Sanity: allow some NaNs; we will mask them during CV per-target
print("✅ Proceeding with per-target masking (NaN labels will be excluded from that target's CV).")


🧪 Label NaN audit:
  - Tg: 7462 / 7973 missing (93.59%)
  - FFV: 943 / 7973 missing (11.83%)
  - Tc: 7236 / 7973 missing (90.76%)
  - Density: 7360 / 7973 missing (92.31%)
  - Rg: 7359 / 7973 missing (92.30%)
✅ Proceeding with per-target masking (NaN labels will be excluded from that target's CV).


In [73]:
# %%
warnings.filterwarnings("ignore")

def normalize_polymer_smiles(s: str) -> str:
    """
    Minimal normalization to keep topology while avoiding RDKit failures:
    - Replace '*' (polymer wildcard) with 'C' (neutral placeholder).
    - Strip whitespace.
    You can extend this if you find more polymer-specific tokens that hurt parsing.
    """
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = s.replace("*", "C")
    return s

def smiles_to_mol(s: str):
    try:
        return Chem.MolFromSmiles(normalize_polymer_smiles(s))
    except Exception:
        return None

train_mols = [smiles_to_mol(s) for s in tqdm(train["SMILES"], desc="RDKit parsing (train)")]
test_mols  = [smiles_to_mol(s) for s in tqdm(test["SMILES"],  desc="RDKit parsing (test)")]

train_parse_rate = np.mean([m is not None for m in train_mols])
test_parse_rate  = np.mean([m is not None for m in test_mols])

# Sanity check
assert train_parse_rate > 0.98, f"Low RDKit parse rate (train): {train_parse_rate:.2%}"
assert test_parse_rate  > 0.98, f"Low RDKit parse rate (test):  {test_parse_rate:.2%}"
print(f"✅ RDKit parse success — train: {train_parse_rate:.2%} | test: {test_parse_rate:.2%}")


RDKit parsing (test): 100%|██████████| 3/3 [00:00<00:00, 537.87it/s]

✅ RDKit parse success — train: 100.00% | test: 100.00%


In [74]:
# %%
warnings.filterwarnings("ignore")

def compute_rdkit_basic(mols):
    rows = []
    for m in tqdm(mols, desc="RDKit basic"):
        if m is None:
            rows.append({
                "MolWt": np.nan, "HeavyAtomCount": np.nan,
                "NumHAcceptors": np.nan, "NumHDonors": np.nan,
                "TPSA": np.nan, "LogP": np.nan, "NumRotatableBonds": np.nan,
                "NumRings": np.nan, "AromaticProportion": np.nan,
            })
            continue

        heavy = Descriptors.HeavyAtomCount(m)
        arom_atoms = sum(int(a.GetIsAromatic()) for a in m.GetAtoms())
        props = {
            "MolWt": Descriptors.MolWt(m),
            "HeavyAtomCount": heavy,
            "NumHAcceptors": Lipinski.NumHAcceptors(m),
            "NumHDonors": Lipinski.NumHDonors(m),
            "TPSA": rdMolDescriptors.CalcTPSA(m),
            "LogP": Crippen.MolLogP(m),
            "NumRotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(m),
            "NumRings": rdMolDescriptors.CalcNumRings(m),
            "AromaticProportion": (arom_atoms / heavy) if heavy > 0 else 0.0,
        }
        rows.append(props)
    return pd.DataFrame(rows)

rdkit_basic_train = compute_rdkit_basic(train_mols)
rdkit_basic_test  = compute_rdkit_basic(test_mols)

# Sanity: same columns & no all-NaN cols
assert list(rdkit_basic_train.columns) == list(rdkit_basic_test.columns)
all_nan_cols = [c for c in rdkit_basic_train.columns if rdkit_basic_train[c].isna().all()]
assert not all_nan_cols, f"All-NaN columns in RDKit basic: {all_nan_cols}"

print("✅ RDKit basic shapes:", rdkit_basic_train.shape, rdkit_basic_test.shape)


RDKit basic: 100%|██████████| 3/3 [00:00<00:00, 1810.23it/s]


✅ RDKit basic shapes: (7973, 9) (3, 9)


In [75]:
# %%
warnings.filterwarnings("ignore")

def morgan_bits_df(mols, radius=2, n_bits=1024):
    data = np.zeros((len(mols), n_bits), dtype=np.uint8)
    for i, m in enumerate(tqdm(mols, desc=f"Morgan r{radius} n{n_bits}")):
        if m is None:
            continue
        bitvect = rdMolDescriptors.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)
        arr = np.zeros((n_bits,), dtype=np.int8)
        Chem.DataStructs.ConvertToNumpyArray(bitvect, arr)
        data[i, :] = arr
    cols = [f"morgan_{radius}_{i}" for i in range(n_bits)]
    return pd.DataFrame(data, columns=cols)

morgan_train = morgan_bits_df(train_mols, radius=2, n_bits=1024)
morgan_test  = morgan_bits_df(test_mols,  radius=2, n_bits=1024)

# Sanity
assert morgan_train.shape[1] == morgan_test.shape[1] == 1024
print("✅ Morgan shapes:", morgan_train.shape, morgan_test.shape)


Morgan r2 n1024: 100%|██████████| 3/3 [00:00<00:00, 4369.07it/s]

✅ Morgan shapes: (7973, 1024) (3, 1024)


In [76]:
# %%
warnings.filterwarnings("ignore")

def physics_features_df(mols):
    rows = []
    for m in tqdm(mols, desc="Physics priors"):
        if m is None:
            rows.append({
                "frac_csp3": np.nan,
                "formal_charge_sum": np.nan,
                "hetero_ratio": np.nan,
                "halogen_count": np.nan,
                "nitrogen_count": np.nan,
                "oxygen_count": np.nan,
                "sulfur_count": np.nan,
                "aromatic_rings": np.nan,
            })
            continue
        heavy = Descriptors.HeavyAtomCount(m)
        frac_csp3 = rdMolDescriptors.CalcFractionCSP3(m)
        formal_charge = sum(a.GetFormalCharge() for a in m.GetAtoms())
        hetero = sum(1 for a in m.GetAtoms() if a.GetAtomicNum() not in (1,6))  # non H/C
        halogens = sum(1 for a in m.GetAtoms() if a.GetAtomicNum() in (9,17,35,53,85))
        n_cnt = sum(1 for a in m.GetAtoms() if a.GetAtomicNum()==7)
        o_cnt = sum(1 for a in m.GetAtoms() if a.GetAtomicNum()==8)
        s_cnt = sum(1 for a in m.GetAtoms() if a.GetAtomicNum()==16)
        arom_rings = rdMolDescriptors.CalcNumAromaticRings(m)

        rows.append({
            "frac_csp3": frac_csp3,
            "formal_charge_sum": formal_charge,
            "hetero_ratio": (hetero / heavy) if heavy > 0 else 0.0,
            "halogen_count": halogens,
            "nitrogen_count": n_cnt,
            "oxygen_count": o_cnt,
            "sulfur_count": s_cnt,
            "aromatic_rings": arom_rings,
        })
    return pd.DataFrame(rows)

phys_train = physics_features_df(train_mols)
phys_test  = physics_features_df(test_mols)

# Sanity
assert list(phys_train.columns) == list(phys_test.columns)
print("✅ Physics priors shapes:", phys_train.shape, phys_test.shape)


Physics priors: 100%|██████████| 3/3 [00:00<00:00, 2431.01it/s]

✅ Physics priors shapes: (7973, 8) (3, 8)


In [77]:
# %%
warnings.filterwarnings("ignore")

def compute_mordred(mols):
    calc = Calculator(descriptors, ignore_3D=True)
    # Mordred returns an iterator of results; convert to DataFrame with tqdm
    rows = []
    for r in tqdm(calc.map(mols), total=len(mols), desc="Mordred"):
        # r is a DescriptorResults; convert to dict safely
        try:
            rows.append(dict(r.asdict()))
        except Exception:
            rows.append({})
    df = pd.DataFrame(rows)
    # Drop non-numeric, inf, all-NaN, and constant cols
    df = df.select_dtypes(include=[np.number])
    df = df.replace([np.inf, -np.inf], np.nan)
    all_nan = [c for c in df.columns if df[c].isna().all()]
    if all_nan:
        df = df.drop(columns=all_nan)
    nunique = df.nunique(dropna=True)
    const_cols = list(nunique[nunique <= 1].index)
    if const_cols:
        df = df.drop(columns=const_cols)
    return df

if MORDRED_ENABLED:
    mordred_train = compute_mordred(train_mols)
    mordred_test  = compute_mordred(test_mols)
    # Align columns in case Mordred returns slightly different sets due to NaNs
    common_cols = sorted(set(mordred_train.columns).intersection(set(mordred_test.columns)))
    mordred_train = mordred_train[common_cols].copy()
    mordred_test  = mordred_test[common_cols].copy()
    print("✅ Mordred shapes:", mordred_train.shape, mordred_test.shape)
else:
    mordred_train = pd.DataFrame(index=np.arange(len(train)))
    mordred_test  = pd.DataFrame(index=np.arange(len(test)))
    print("⚠️ Mordred unavailable — skipping (set MORDRED_ENABLED=True if installed)")

# Sanity
assert mordred_train.shape[1] == mordred_test.shape[1], "Train/Test Mordred column mismatch"


Mordred: 100%|██████████| 3/3 [00:00<00:00,  3.44it/s]


✅ Mordred shapes: (7973, 888) (3, 888)


In [78]:
# %%
warnings.filterwarnings("ignore")

# Choose which blocks to include in the final model
FEATURE_BLOCKS = {
    "rdkit_basic": (rdkit_basic_train, rdkit_basic_test),
    "morgan": (morgan_train, morgan_test),
    "physics": (phys_train, phys_test),
    "mordred": (mordred_train, mordred_test),  # may be empty if disabled
}

def concat_blocks(blocks_dict):
    train_parts, test_parts = [], []
    for name, (df_tr, df_te) in blocks_dict.items():
        # Align columns just in case
        if df_tr.shape[1] and df_te.shape[1]:
            common = sorted(set(df_tr.columns).intersection(df_te.columns))
            df_tr = df_tr[common].reset_index(drop=True)
            df_te = df_te[common].reset_index(drop=True)
        train_parts.append(df_tr.values if df_tr.shape[1] else np.zeros((len(df_tr), 0)))
        test_parts.append(df_te.values if df_te.shape[1] else np.zeros((len(df_te), 0)))
    Xtr = np.hstack(train_parts)
    Xte = np.hstack(test_parts)
    return Xtr, Xte

X_train_raw, X_test_raw = concat_blocks(FEATURE_BLOCKS)

# Impute NaNs (median)
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train_raw)
X_test  = imputer.transform(X_test_raw)

# Sanity
assert X_train.shape[0] == len(train)
assert X_test.shape[0]  == len(test)
print("✅ Final features:", X_train.shape, X_test.shape)
print("Blocks used & dims:")
for name, (df_tr, _) in FEATURE_BLOCKS.items():
    print(f"  - {name:<12}: {df_tr.shape[1]:>5} cols")


✅ Final features: (7973, 1929) (3, 1929)
Blocks used & dims:
  - rdkit_basic :     9 cols
  - morgan      :  1024 cols
  - physics     :     8 cols
  - mordred     :   888 cols


In [82]:
# %%
# Patch B — Masked mMAE helper (ignores NaN labels per target)
def compute_mmae_masked(y_true_df, y_pred_df, targets):
    maes = []
    per_target = {}
    for t in targets:
        mask = ~y_true_df[t].isna()
        if mask.sum() == 0:
            # No labels for this target — skip (won't happen in practice)
            continue
        mae_t = mean_absolute_error(y_true_df.loc[mask, t].values,
                                    y_pred_df.loc[mask, t].values)
        maes.append(mae_t)
        per_target[t] = float(mae_t)
    return (float(np.mean(maes)) if maes else float("nan")), per_target

print("✅ Masked mMAE helper ready.")


✅ Masked mMAE helper ready.


In [86]:
# %%
# Cell 11 — Adaptive CV training with real-time mMAE per target
import time, warnings
warnings.filterwarnings("ignore")

BASE_LGB_PARAMS = dict(
    objective="mae",
    boosting_type="gbdt",
    learning_rate=0.05,
    n_estimators=2000,        # overridden adaptively
    num_leaves=64,            # overridden adaptively
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    min_data_in_leaf=20,
    reg_alpha=0.0,
    reg_lambda=0.0,
    verbose=-1,
    random_state=SEED,
    n_jobs=-1,
)

def choose_cv_and_params(n_labeled: int):
    if n_labeled >= 2000:
        n_folds, est, stop, leaves = 5, 2000, 100, 64
    elif n_labeled >= 750:
        n_folds, est, stop, leaves = 5, 1500, 80, 48
    elif n_labeled >= 300:
        n_folds, est, stop, leaves = 4, 1200, 70, 36
    elif n_labeled >= 150:
        n_folds, est, stop, leaves = 3, 1000, 60, 24
    else:
        n_folds, est, stop, leaves = 2, 700, 40, 16
    return n_folds, est, stop, leaves

from sklearn.model_selection import KFold
import lightgbm as lgb
from tqdm import tqdm

y = train[EXPECTED_TARGETS].copy()

oof = pd.DataFrame(np.nan, index=train.index, columns=EXPECTED_TARGETS)
test_pred = pd.DataFrame(0.0,     index=test.index,  columns=EXPECTED_TARGETS)

overall_start = time.time()
per_target_times = {}
per_target_mae = {}

for t in tqdm(EXPECTED_TARGETS, desc="Targets"):
    valid_idx = np.where(~y[t].isna())[0]
    n_lab = len(valid_idx)

    if n_lab < 20:
        print(f"⚠️ Target {t}: only {n_lab} labeled rows — skipping (preds remain 0).")
        continue

    n_folds_t, est, stop_rounds, leaves = choose_cv_and_params(n_lab)
    kf_t = KFold(n_splits=n_folds_t, shuffle=True, random_state=SEED)

    print(f"\nTarget {t}: {n_lab} labeled rows → {n_folds_t} folds, "
          f"{est} estimators, ES={stop_rounds}, num_leaves={leaves}")

    params_t = dict(BASE_LGB_PARAMS)
    params_t.update(dict(n_estimators=est, num_leaves=leaves))

    preds_test_folds = []
    t_start = time.time()

    for fold, (tr_sub, va_sub) in enumerate(kf_t.split(valid_idx), start=1):
        fold_start = time.time()
        tr_idx = valid_idx[tr_sub]
        va_idx = valid_idx[va_sub]

        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y.loc[tr_idx, t].values, y.loc[va_idx, t].values

        model = lgb.LGBMRegressor(**params_t)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="l1",
            callbacks=[lgb.early_stopping(stopping_rounds=stop_rounds, verbose=False)],
        )

        oof.loc[va_idx, t] = model.predict(X_va, num_iteration=model.best_iteration_)
        preds_test_folds.append(model.predict(X_test, num_iteration=model.best_iteration_))

        print(f"  Fold {fold}/{n_folds_t} | best_iter={model.best_iteration_} "
              f"| val_size={len(va_idx)} | time={time.time()-fold_start:.1f}s")

    test_pred[t] = np.mean(np.column_stack(preds_test_folds), axis=1)
    per_target_times[t] = time.time() - t_start

    # ✅ Compute per-target masked MAE right after finishing this target
    mask = ~y[t].isna()
    mae_t = mean_absolute_error(y.loc[mask, t].values, oof.loc[mask, t].values)
    per_target_mae[t] = mae_t
    print(f"✓ Target {t} done in {per_target_times[t]:.1f}s "
          f"(~{per_target_times[t]/n_folds_t:.1f}s/fold) | MAE={mae_t:.5f}")

# ✅ Final masked mMAE across all targets
cv_mmae, per_t = compute_mmae_masked(y, oof, EXPECTED_TARGETS)
print("\n================ CV Summary ================")
print("Masked CV mMAE:", f"{cv_mmae:.5f}")
print("Per-target MAE:", {k: round(v, 5) for k, v in per_target_mae.items()})
print("Per-target times (s):", {k: round(v,1) for k, v in per_target_times.items()})
print(f"Total training time: {time.time()-overall_start:.1f}s")

# Sanity checks
for t in EXPECTED_TARGETS:
    mask = ~oof[t].isna()
    if mask.any():
        assert np.isfinite(oof.loc[mask, t]).all(), f"Non-finite OOF in {t}"
    assert np.isfinite(test_pred[t]).all(), f"Non-finite Test Pred in {t}"
print("✅ Training complete & sanity checks passed.")


Targets:   0%|          | 0/5 [00:00<?, ?it/s]


Target Tg: 511 labeled rows → 4 folds, 1200 estimators, ES=70, num_leaves=36
  Fold 1/4 | best_iter=368 | val_size=128 | time=5.1s
  Fold 2/4 | best_iter=621 | val_size=128 | time=7.9s
  Fold 3/4 | best_iter=219 | val_size=128 | time=1.1s


Targets:  20%|██        | 1/5 [00:14<00:59, 14.81s/it]

  Fold 4/4 | best_iter=147 | val_size=127 | time=0.7s
✓ Target Tg done in 14.8s (~3.7s/fold) | MAE=50.89446

Target FFV: 7030 labeled rows → 5 folds, 2000 estimators, ES=100, num_leaves=64


Targets:  20%|██        | 1/5 [00:22<01:28, 22.19s/it]


KeyboardInterrupt: 

In [87]:
mask = ~train["Tg"].isna()
tg_vals = train.loc[mask, "Tg"]
print("Tg median:", tg_vals.median())
print("Tg IQR:", tg_vals.quantile(0.75) - tg_vals.quantile(0.25))
print("Baseline MAE (predict mean):", mean_absolute_error(tg_vals, np.full_like(tg_vals, tg_vals.mean())))


Tg median: 74.04018308
Tg IQR: 147.47308598
Baseline MAE (predict mean): 88.0556268278731


In [88]:
# %%
import os

def list_files_recursive(base_path, max_depth=3):
    for root, dirs, files in os.walk(base_path):
        depth = root[len(base_path):].count(os.sep)
        if depth < max_depth:
            indent = "  " * depth
            print(f"{indent}{os.path.basename(root)}/")
            subindent = "  " * (depth + 1)
            for f in files:
                print(f"{subindent}{f}")

print("📂 PI1M-master:")
list_files_recursive("PI1M-master", max_depth=3)

print("\n📂 Lieconv-Tg-main:")
list_files_recursive("Lieconv-Tg-main", max_depth=3)


📂 PI1M-master:
PI1M-master/
  sa_PI1M.png
  PI1M.csv
  PI1M_v2.csv
  LICENSE
  README.md

📂 Lieconv-Tg-main:
Lieconv-Tg-main/
  LICENSE
  README.md
  lieconv.png
  Model training and Predict/
    Lieconv/
      Lieconv-training-files.ipynb
      lieconv training and predict.py
    Image-cnn/
      Image-CNN training and predict.py
      Image-CNN-trainingfiles.ipynb
    ECC/
      ECC-training-files.ipynb
      ECC training and predict.py
  requirements/
    conda/
      lieconv-environment.yml
      ecc-environment.yml
      image-cnn-environment.yml
  wandb/
    Lieconv/
      wandb_lieconv.py
    Image-CNN/
      wandb_image-cnn.py
    ECC/
      wandb_ecc.py
  models/
    Lieconv/
      lieconv_model.pkl
    Image-CNN/
      image-cnn.hdf5
    ECC/
  polymer screening/
    PI1M/
      Screening example.ipynb
      PI1M_MF_2.csv
      PI1M_MF_3_with_morgan_fingerprint_group.csv
      PI1M_MF_1.csv
      PI1M_MF_2_with_morgan_fingerprint_group.csv
      PI1M_MF_1_with_morgan_fingerpr

In [91]:
# %%
import pandas as pd
import os

# Path to your root directory
ROOT_DIR = "."

# Find all CSVs in root (not subfolders)
csv_files = [f for f in os.listdir(ROOT_DIR) if f.endswith(".csv")]

print(f"Found {len(csv_files)} CSV files in root dir:\n", csv_files, "\n")

for f in csv_files:
    path = os.path.join(ROOT_DIR, f)
    try:
        df = pd.read_csv(path)
        print("="*80)
        print(f"📄 {f}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        print("  First 5 rows:")
        display(df.head())
    except Exception as e:
        print(f"⚠️ Could not read {f}: {e}")


Found 21 CSV files in root dir:
 ['train.csv', 'sample_submission.csv', 'dataset4.csv', 'dataset1.csv', 'PolyAskInG_3_Tg_greater than_300.csv', 'PI1M.csv', 'PI1M_v2.csv', 'submission_fixed.csv', 'PI1M_MF_2.csv', 'submission.csv', 'JCIM_sup_bigsmiles.csv', 'PI1M_MF_3_with_morgan_fingerprint_group.csv', 'dataset3.csv', 'Bicerano_bigsmiles.csv', 'PolyAskInG_3_Tg_greater than_400.csv', 'test.csv', 'PI1M_MF_1.csv', 'PI1M_MF_2_with_morgan_fingerprint_group.csv', 'PI1M_MF_1_with_morgan_fingerprint_group.csv', 'PI1M_MF_3.csv', 'dataset2.csv'] 

📄 train.csv
  Shape: (7973, 7)
  Columns: ['id', 'SMILES', 'Tg', 'FFV', 'Tc', 'Density', 'Rg']
  First 5 rows:


,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,NaN,0.370410,NaN,NaN,NaN
2,388772,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,NaN,0.378860,NaN,NaN,NaN
3,519416,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,NaN,0.387324,NaN,NaN,NaN
4,539187,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,NaN,0.355470,NaN,NaN,NaN


📄 sample_submission.csv
  Shape: (3, 6)
  Columns: ['id', 'Tg', 'FFV', 'Tc', 'Density', 'Rg']
  First 5 rows:


,id,Tg,FFV,Tc,Density,Rg
0,1109053969,0,0,0,0,0
1,1422188626,0,0,0,0,0
2,2032016830,0,0,0,0,0


📄 dataset4.csv
  Shape: (862, 2)
  Columns: ['SMILES', 'FFV']
  First 5 rows:


...

,index,Smiles,SA,Tg_pred,Morgan fingerprint pairs
0,8863,*C(*)=O,6.166200,2565.980957,[264]
1,2834,*C(*)=S,6.707931,2176.972168,[264]
2,4182,*C(*)=C=N,6.972416,1365.365845,[264]
3,5829,*C(*)=C([2H])Cl,7.174129,1057.386353,[264]
4,7590,*C(*)=C(Cl)Cl,5.808430,1009.497131,[264]


📄 PI1M_MF_3.csv
  Shape: (440232, 5)
  Columns: ['index', 'Smiles', 'SA', 'Tg_true', 'Tg_pred']
  First 5 rows:


,index,Smiles,SA,Tg_true,Tg_pred
0,6880,*c1ccc(-c2ccc(-c3nc4ccc(-c5ccc6nc(*)sc6c5)cc4s...,3.148073,1,578.546692
1,881,*c1ccc(-c2nc3ccc(-c4ccc5nc(*)[nH]c5c4)cc3o2)cc1,3.315860,1,496.296509
2,7442,*c1ccc(-c2ccc(-c3nc4cc5nc(*)[nH]c5cc4s3)cc2)cc1,3.375013,1,494.074524
3,4716,*c1ccc(-c2ccc(-c3nc4ccc(-c5ccc6nc(*)sc6c5)cc4o...,3.161383,1,478.144348
4,6703,*c1cccc(-c2nc3ccc(-c4ccc5nc(*)[nH]c5c4)cc3o2)c1,3.324460,1,443.563019


📄 dataset2.csv
  Shape: (7208, 1)
  Columns: ['SMILES']
  First 5 rows:


,SMILES
0,*/C(=C(/*)c1ccc(C(C)(C)C)cc1)c1ccccc1
1,*/C(=C(/*)c1ccc(CCCC)cc1)c1ccccc1
2,*/C(=C(/*)c1ccc(Oc2ccccc2)cc1)c1ccccc1
3,*/C(=C(/*)c1ccc([Si](C(C)C)(C(C)C)C(C)C)cc1)c1...
4,*/C(=C(/*)c1ccc([Si](C)(C)C)cc1)c1ccccc1


In [92]:
import os
import shutil
from pathlib import Path

# Define source and target folders
src_dir = Path(".")  # current directory
comp_dir = Path("competition")
ext_dir = Path("external")

# Create target folders if they don’t exist
comp_dir.mkdir(exist_ok=True)
ext_dir.mkdir(exist_ok=True)

# Keywords that identify competition files
competition_keywords = ["train", "test", "sample_submission"]

# Loop through CSV files and sort
for csv_file in src_dir.glob("*.csv"):
    fname = csv_file.name.lower()
    if any(key in fname for key in competition_keywords):
        shutil.move(str(csv_file), comp_dir / csv_file.name)
        print(f"Moved {csv_file} → {comp_dir}/")
    else:
        shutil.move(str(csv_file), ext_dir / csv_file.name)
        print(f"Moved {csv_file} → {ext_dir}/")


Moved train.csv → competition/
Moved sample_submission.csv → competition/
Moved dataset4.csv → external/
Moved dataset1.csv → external/
Moved PolyAskInG_3_Tg_greater than_300.csv → external/
Moved PI1M_v2.csv → external/
Moved submission_fixed.csv → external/
Moved submission.csv → external/
Moved JCIM_sup_bigsmiles.csv → external/
Moved dataset3.csv → external/
Moved Bicerano_bigsmiles.csv → external/
Moved PolyAskInG_3_Tg_greater than_400.csv → external/
Moved test.csv → competition/


## Preprocessing 

In [95]:
# Core
import numpy as np
import pandas as pd

# ML
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# Chemistry
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# Utils
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Competition metric: mean MAE across targets
def mMAE(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


In [96]:
# Load only the competition data
train = pd.read_csv("competition/train.csv")
test = pd.read_csv("competition/test.csv")
sample = pd.read_csv("competition/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

targets = [c for c in train.columns if c not in ["id", "SMILES"]]
print("Targets:", targets)


Train shape: (7973, 7)
Test shape: (3, 2)
Targets: ['Tg', 'FFV', 'Tc', 'Density', 'Rg']


In [97]:
def featurize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.Series(dtype=float)

    return pd.Series({
        "MolWt": Descriptors.MolWt(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumRotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "NumHAcceptors": rdMolDescriptors.CalcNumHBA(mol),
        "NumHDonors": rdMolDescriptors.CalcNumHBD(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
    })

# Generate features for train and test
train_feats = train["SMILES"].apply(featurize)
test_feats  = test["SMILES"].apply(featurize)

print("Feature matrix:", train_feats.shape)


Feature matrix: (7973, 8)


In [98]:
train_full = pd.concat([train_feats, train[targets]], axis=1)
test_full  = test_feats.copy()

print("Final train shape:", train_full.shape)
print("Final test shape:", test_full.shape)


Final train shape: (7973, 13)
Final test shape: (3, 8)


In [99]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

params = {
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 64,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbosity": -1,
}


In [103]:
import numpy as np
import pandas as pd

# Drop ID/SMILES if still in frame
X_features = train_full.drop(columns=targets, errors="ignore")

print("=== Feature Audit ===")
print(f"Total features: {X_features.shape[1]}")
print(f"Total samples: {X_features.shape[0]}")

# 1. Constant features
n_const = (X_features.nunique() <= 1).sum()
print(f"Constant features: {n_const}")

# 2. Feature variance
variances = X_features.var().sort_values()
print(f"Low variance features (<1e-6): {(variances < 1e-6).sum()}")

# 3. Top correlated features with each target
print("\n=== Top correlations with targets ===")
for target in targets:
    if target not in train_full.columns: 
        continue
    y = train_full[target]
    mask = ~y.isna()
    if mask.sum() == 0:
        print(f"⚠️ {target}: no labels available.")
        continue
    
    corrs = X_features.loc[mask].corrwith(y[mask])
    top_corrs = corrs.abs().sort_values(ascending=False).head(10)
    
    print(f"\nTarget: {target} (n={mask.sum()} labeled)")
    for feat in top_corrs.index:
        print(f"  {feat}: corr={corrs[feat]:.3f}")


=== Feature Audit ===
Total features: 8
Total samples: 7973
Constant features: 0
Low variance features (<1e-6): 0

=== Top correlations with targets ===

Target: Tg (n=511 labeled)
  RingCount: corr=0.629
  FractionCSP3: corr=-0.610
  NumRotatableBonds: corr=-0.321
  HeavyAtomCount: corr=0.282
  MolWt: corr=0.251
  TPSA: corr=0.213
  NumHAcceptors: corr=0.147
  NumHDonors: corr=0.112

Target: FFV (n=7030 labeled)
  NumHDonors: corr=-0.331
  RingCount: corr=0.232
  TPSA: corr=-0.216
  MolWt: corr=0.162
  HeavyAtomCount: corr=0.157
  NumHAcceptors: corr=-0.148
  FractionCSP3: corr=-0.075
  NumRotatableBonds: corr=-0.026

Target: Tc (n=737 labeled)
  NumRotatableBonds: corr=0.723
  HeavyAtomCount: corr=0.465
  MolWt: corr=0.440
  NumHDonors: corr=0.424
  FractionCSP3: corr=0.332
  TPSA: corr=0.233
  RingCount: corr=-0.105
  NumHAcceptors: corr=0.084

Target: Density (n=613 labeled)
  NumHAcceptors: corr=0.344
  FractionCSP3: corr=-0.343
  NumRotatableBonds: corr=-0.302
  TPSA: corr=0.260


In [101]:
from lightgbm import early_stopping, log_evaluation

oof = np.zeros((len(train_full), len(targets)))
preds = np.zeros((len(test_full), len(targets)))

for t, target in enumerate(targets):
    print(f"\n=== Training target: {target} ===")
    y = train_full[target].values
    X = train_full.drop(columns=targets).values
    X_test = test_full.values

    # Skip if target is all NaN
    if np.all(np.isnan(y)):
        print(f"⚠️ Skipping {target}, all values missing.")
        continue

    for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f"  Fold {fold+1}/{kf.n_splits}")
        X_train, y_train = X[trn_idx], y[trn_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        train_data = lgb.Dataset(X_train, label=y_train)
        val_data   = lgb.Dataset(X_val, label=y_val)

        model = lgb.train(
            params,
            train_data,
            valid_sets=[train_data, val_data],
            num_boost_round=2000,
            callbacks=[
                early_stopping(stopping_rounds=100, verbose=False),
                log_evaluation(0),
            ]
        )

        val_pred = model.predict(X_val, num_iteration=model.best_iteration)
        test_pred = model.predict(X_test, num_iteration=model.best_iteration)

        oof[val_idx, t] = val_pred
        preds[:, t] += test_pred / kf.n_splits

    # Compute metrics
    mask = ~np.isnan(y)
    model_mae = mMAE(y[mask], oof[mask, t])
    baseline_pred = np.full_like(y[mask], np.nanmean(y[mask]))
    baseline_mae = mMAE(y[mask], baseline_pred)

    print(f"✓ Target {target} done | Model mMAE={model_mae:.4f} | Baseline MAE={baseline_mae:.4f}")



=== Training target: Tg ===
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
✓ Target Tg done | Model mMAE=100.1468 | Baseline MAE=88.0556

=== Training target: FFV ===
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
✓ Target FFV done | Model mMAE=0.0375 | Baseline MAE=0.0209

=== Training target: Tc ===
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
✓ Target Tc done | Model mMAE=0.1191 | Baseline MAE=0.0770

=== Training target: Density ===
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
✓ Target Density done | Model mMAE=0.6536 | Baseline MAE=0.1094

=== Training target: Rg ===
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5
✓ Target Rg done | Model mMAE=10.5426 | Baseline MAE=3.9627


In [102]:
overall = mMAE(train_full[targets].values, oof)
print("Overall CV mMAE:", overall)


Overall CV mMAE: nan


# Preprocessing 2.0

In [116]:
# Cell 1: Imports and setup
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
from tqdm import tqdm
import os

# Sanity: Print versions
print("RDKit / LightGBM / Pandas versions:")
print(pd.__version__)
print(lgb.__version__)


RDKit / LightGBM / Pandas versions:
2.3.1
4.6.0


In [118]:
# Cell 2: Data loading
train = pd.read_csv("competition/train.csv")
test = pd.read_csv("competition/test.csv")
sample = pd.read_csv("competition/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample sub shape:", sample.shape)

# Sanity: check for nulls
print("Train nulls:", train.isnull().sum().sum())
print("Test nulls:", test.isnull().sum().sum())


Train shape: (7973, 7)
Test shape: (3, 2)
Sample sub shape: (3, 6)
Train nulls: 30360
Test nulls: 0


In [119]:
# Cell 3: Feature builder with error handling
def build_features(df):
    feats = []
    for smi in df['SMILES']:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            # Handle invalid SMILES safely
            feats.append([np.nan] * 6)
            continue
        
        # Physics-informed features
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        frac_csp3 = Descriptors.FractionCSP3(mol)
        mol_wt = Descriptors.MolWt(mol)
        tpsa = Descriptors.TPSA(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        
        feats.append([rot_bonds, frac_csp3, mol_wt, tpsa, hbd, hba])
    
    feats = np.array(feats)
    colnames = ["rot_bonds", "frac_csp3", "mol_wt", "tpsa", "hbd", "hba"]
    return pd.DataFrame(feats, columns=colnames)

# Build features
X_train = build_features(train)
X_test = build_features(test)

# Merge back with IDs
X_train['id'] = train['id']
X_test['id'] = test['id']
y_train = train[['Tg', 'Rg']]

print("Feature matrix shape:", X_train.shape, X_test.shape)
print("Example features:\n", X_train.head())


Feature matrix shape: (7973, 7) (3, 7)
Example features:
    rot_bonds  frac_csp3    mol_wt    tpsa  hbd   hba      id
0        8.0   0.533333   232.323   26.30  0.0   2.0   87817
1       16.0   0.441860   598.919   24.06  2.0   2.0  106919
2       15.0   0.145161  1003.207  122.27  0.0   9.0  388772
3        7.0   0.100000   542.726   24.06  2.0   2.0  519416
4       34.0   0.518519   965.154  182.28  0.0  14.0  539187


In [120]:
# Cell 4: Custom competition metric (mean MAE across targets)
def mmae_score(y_true, y_pred):
    if isinstance(y_pred, pd.DataFrame):
        y_pred = y_pred.values
    if isinstance(y_true, pd.DataFrame):
        y_true = y_true.values
    
    # Sanity: ensure correct shapes
    assert y_true.shape == y_pred.shape, f"Shape mismatch {y_true.shape} vs {y_pred.shape}"
    
    maes = []
    for i in range(y_true.shape[1]):
        m = mean_absolute_error(y_true[:, i], y_pred[:, i])
        maes.append(m)
    return np.mean(maes)

# Sanity check
yt = np.array([[1,2],[2,3]])
yp = np.array([[1,2.1],[2.2,2.9]])
print("Test MMAE:", mmae_score(yt, yp))


Test MMAE: 0.10000000000000009


In [123]:
# Cell 5 (fixed again): Safe CV training with NaN handling
def train_model(X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    models = []
    oof = np.full(y.shape, np.nan)  # keep NaNs for missing labels
    
    for fold, (trn_idx, val_idx) in enumerate(kf.split(X), 1):
        print(f"\n---- Fold {fold} ----")
        X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]
        
        preds_val = np.full(y_val.shape, np.nan)
        
        for j, target in enumerate(['Tg', 'Rg']):
            # Skip if no labels in train/val for this fold
            if y_tr[target].isna().all() or y_val[target].isna().all():
                print(f"⚠️ Skipping target {target} (no labels in fold)")
                continue
            
            lgb_train = lgb.Dataset(
                X_tr.drop(columns=['id']),
                y_tr[target],
                free_raw_data=False
            )
            lgb_val = lgb.Dataset(
                X_val.drop(columns=['id']),
                y_val[target],
                reference=lgb_train,
                free_raw_data=False
            )
            
            params = {
                "objective": "regression",
                "metric": "mae",
                "verbosity": -1,
                "learning_rate": 0.05,
                "num_leaves": 31,
                "seed": 42,
            }
            
            model = lgb.train(
                params,
                lgb_train,
                valid_sets=[lgb_val],
                num_boost_round=2000,
                callbacks=[
                    lgb.early_stopping(100),
                    lgb.log_evaluation(200)
                ]
            )
            
            val_pred = model.predict(
                X_val.drop(columns=['id']),
                num_iteration=model.best_iteration
            )
            preds_val[:, j] = val_pred
            models.append(model)
            
            # ✅ Safe MAE calculation only on non-NaN labels
            mask = ~y_val[target].isna()
            fold_mae = mean_absolute_error(
                y_val[target][mask],
                val_pred[mask]
            )
            print(f"   Target {target} | Fold {fold} MAE={fold_mae:.4f}")
        
        # Save fold preds
        oof[val_idx] = preds_val
    
    # Compute overall MMAE safely
    mask = ~np.isnan(y.values) & ~np.isnan(oof)
    maes = []
    for j in range(y.shape[1]):
        col_mask = mask[:, j]
        if col_mask.any():
            maes.append(mean_absolute_error(y.values[col_mask, j], oof[col_mask, j]))
    oof_score = np.mean(maes)
    print("\nOverall OOF MMAE:", oof_score)
    
    return models, oof

# ✅ Run safely
models, oof_preds = train_model(X_train, y_train)



---- Fold 1 ----
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[76]	valid_0's l1: 10.735
   Target Tg | Fold 1 MAE=90.5141
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 1.6183
Early stopping, best iteration is:
[126]	valid_0's l1: 1.59544
   Target Rg | Fold 1 MAE=10.4475

---- Fold 2 ----
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[62]	valid_0's l1: 12.3107
   Target Tg | Fold 2 MAE=107.4618
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 1.65286
Early stopping, best iteration is:
[118]	valid_0's l1: 1.63428
   Target Rg | Fold 2 MAE=10.1171

---- Fold 3 ----
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[52]	valid_0's l1: 12.8507
   Target Tg | Fold 3 MAE=104.1920
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 1.68825
Early st

In [124]:
# Cell 6: Inference
def infer(models, X):
    n_targets = 2
    preds = np.zeros((X.shape[0], n_targets))
    
    # Average over folds
    for i, model in enumerate(models):
        target_idx = i % n_targets
        preds[:, target_idx] += model.predict(X.drop(columns=['id']), num_iteration=model.best_iteration)
    
    preds /= (len(models) / n_targets)
    return preds

test_preds = infer(models, X_test)
print("Inference shape:", test_preds.shape)


Inference shape: (3, 2)
